# Урок 5.1. Когда локальные LLM — не просто «можно», а «нужно»

Модуль 5 «Локальные LLM: запуск без облака и без GPU» · урок 1 из 7

Ещё в уроке 1.3 мы сравнивали облачные API и локальные модели «на глаз»:
цены за токены против бесплатных весов. Весь курс с тех пор работает на локальной
qwen2.5:3b — но выбор «локально» мы приняли за вас, потому что курс должен быть
бесплатным. Пора вернуть долг: разобраться, **когда такой выбор правилен в бою**,
а когда честнее платить за API.

## 🎯 Что вы узнаете

- Что технически означает «локальная LLM» и чем она отличается от облачного API
- Четыре причины, по которым локальный запуск становится обязательным: приватность, комплаенс, стоимость, offline
- Сколько на самом деле стоят облачные токены и локальное железо — с калькулятором и точкой безубыточности
- Честные минусы локальных моделей: разрыв в качестве, скорость на CPU, операционная нагрузка
- Как принимать решение «локально или API» по взвешенной матрице, а не по ощущениям


## Где мы в курсе

Модули 1–4 научили строить: промпты, эмбеддинги, RAG, защиту и мониторинг.
Модуль 5 — про **фундамент, на котором всё это стояло**: локальную модель.
До сих пор Ollama была для нас чёрным ящиком из урока 1.5 — «поставили и работает».
Теперь разберём её как инженеры: от решения «а нужна ли она вообще» до
production-обвязки.

| Урок | Вопрос, на который отвечает |
|---|---|
| **5.1 (сейчас)** | Когда локальная LLM — необходимость, а не хобби? |
| 5.2 | Какую модель и квантизацию выбрать под мою RAM и мои задачи? |
| 5.3 | Как превратить «запустил в терминале» в надёжный сервис? |
| 5.4 | Что под капотом Ollama: прямой inference через llama-cpp-python |
| 5.5 | Как мониторить: токены/сек, память, деградация, алерты |
| 5.6 | Что делать, когда всё сломалось: OOM, зависания, конкуренция |
| 5.7 | Закрепление: FastAPI-прокси перед Ollama |

В этом уроке кода с моделью не будет вовсе — только расчёты на чистом Python.
Решение «локально или облако» принимается до первого `ollama pull`, а не после.


## Что такое «локальная LLM» технически

LLM — это, грубо говоря, огромный файл с числами (веса нейросети) плюс программа, которая умеет по этим числам предсказывать следующий токен. Когда вы дёргаете облачный API (OpenAI, Anthropic, YandexGPT), файл и программа живут в чужом дата-центре, а вы шлёте туда свои данные по HTTPS.

«Локальная LLM» означает: **файл с весами лежит на вашем диске, а предсказание (inference) выполняет ваш процессор**. Данные не покидают ваш компьютер или ваш контур.

Аналогия для backend-разработчика — базы данных:

| | Облачный API (OpenAI и т.п.) | Локальная LLM (Ollama, llama.cpp) |
|---|---|---|
| Аналог из мира БД | Управляемый облачный сервис (RDS, MongoDB Atlas) | Свой PostgreSQL в докере / SQLite в приложении |
| Кто отвечает за железо и обновления | Провайдер | Вы |
| Куда уходят данные | В чужой дата-центр | Никуда |
| Оплата | За использование (за токены) | Капитальные затраты: железо + электричество + ваше время |
| Качество топовых моделей | Максимальное на рынке | Ниже: вы ограничены открытыми весами и своей RAM |

Путь запроса наглядно:

```
ОБЛАЧНЫЙ API:
  ваш код --HTTPS--> интернет --> дата-центр провайдера --> модель 1000B+
           [данные покидают контур]        [оплата за каждый токен]

ЛОКАЛЬНО:
  ваш код --HTTP--> 127.0.0.1:11434 --> ollama --> файл модели 1-8B на вашем диске
           [данные не выходят никуда]   [оплата: RAM, ватты и ваше время]
```

Ключевое следствие: локальная модель — это не «бесплатный ChatGPT», а **другой инженерный компромисс**. Дальше разберём, когда он выгоден.

## Причина 1: приватность данных

Как только промпт ушёл по сети, вы **юридически и технически теряете над ним контроль**. Даже если провайдер обещает не обучаться на ваших данных (у серьёзных провайдеров для API это так и есть — важно быть честным), остаются:

- логи и ретеншн на стороне провайдера (обычно 0–30 дней, зависит от договора);
- субподрядчики и юрисдикции, через которые проходит трафик;
- человеческий фактор: разработчик случайно вставил в промпт дамп таблицы с персональными данными — и он уже «уехал».

Локальная модель снимает весь класс проблем сразу: некуда утекать. Это то же самое ощущение, что и разница между «положил файл в S3 чужого аккаунта» и «файл лежит на моём сервере в стойке».

## Причина 2: комплаенс и регуляторика

Иногда решение принимаете не вы, а закон или служба безопасности:

- **Персональные данные**: 152-ФЗ в России, GDPR в Европе — трансграничная передача ПДн требует отдельных оснований, а часто просто запрещена политикой компании.
- **Отраслевые тайны**: банковская тайна, врачебная тайна, гостайна, коммерческая тайна под NDA.
- **Сертифицированные контуры**: закрытые сети без доступа в интернет (air-gapped), где облачный API физически недоступен.

В этих случаях матрица решения вырождается: у вас **жёсткое ограничение (hard constraint)**, и вопрос не «что лучше», а «какая локальная модель справится». Типичный сценарий из банковской практики: классификация обращений клиентов, где в текстах ФИО и номера счетов, — только локально или в собственном контуре.

## Причина 3: стоимость на больших объёмах

Облачные API тарифицируются за токены (для английского токен — примерно ¾ слова; русский текст дробится на токены заметно мельче — мы видели это в уроке 1.1). Пока у вас 100 запросов в день — это копейки. Но токены масштабируются линейно с нагрузкой, а железо — ступенькой: сервер, купленный один раз, обрабатывает и 10-й, и 10 000-й запрос за те же деньги (пока хватает пропускной способности).

Считать надо честно, в обе стороны — калькулятор ниже. Спойлер: **на CPU выгода по деньгам появляется реже, чем кажется**, потому что CPU генерирует медленно, и «пропускная способность» вашего сервера ограничена. Зато для задач с длинным входом и коротким выходом (классификация, извлечение полей) CPU-модель может быть очень выгодной.

## Причина 4: offline и предсказуемая задержка

- Приложение должно работать **без интернета**: полевые устройства, суда, производственные площадки, самолёты.
- Нужна **предсказуемость**: нет чужих rate limit, нет деградации у провайдера в час пик, нет «API вернул 529, повторите позже». Ваша модель — ваш SLA.
- Разработка и CI: гонять тесты по API дорого и медленно, а локальная маленькая модель отвечает бесплатно и стабильно (temperature=0 + seed).

## Считаем деньги: API против своего железа

Напишем простой калькулятор. Модель расчёта:

- **API**: платим за входные и выходные токены. Цены возьмём из таблицы урока 1.3 —
  сравним сразу два ценовых класса: бюджетный (`gpt-4o-mini`: $0.15 / $0.60 за 1 млн
  входных/выходных токенов) и топовый (`claude-sonnet`: $3.00 / $15.00).
- **Локально**: железо амортизируем на 36 месяцев, добавляем электричество и немного
  времени инженера на поддержку.

Обратите внимание: у локального варианта есть **потолок производительности** —
CPU-сервер физически не сгенерирует больше `tok_per_sec * секунд_в_месяце` токенов.


In [ ]:
from dataclasses import dataclass


@dataclass
class Workload:
    """Профиль нагрузки: сколько и каких токенов мы гоняем в день."""
    requests_per_day: int
    input_tokens_per_request: int
    output_tokens_per_request: int

    @property
    def monthly_input_tokens(self) -> float:
        return self.requests_per_day * self.input_tokens_per_request * 30

    @property
    def monthly_output_tokens(self) -> float:
        return self.requests_per_day * self.output_tokens_per_request * 30


def api_monthly_cost(w: Workload, price_in_per_m: float, price_out_per_m: float) -> float:
    """Стоимость нагрузки через облачный API, $/месяц.

    price_in_per_m / price_out_per_m — цена за 1 млн входных/выходных токенов.
    """
    return (w.monthly_input_tokens / 1e6 * price_in_per_m
            + w.monthly_output_tokens / 1e6 * price_out_per_m)


def local_monthly_cost(hardware_price: float = 1200.0,
                       amortization_months: int = 36,
                       avg_power_watts: float = 120.0,
                       kwh_price: float = 0.10,
                       ops_hours_per_month: float = 2.0,
                       engineer_hourly_rate: float = 30.0) -> float:
    """Стоимость своего CPU-сервера, $/месяц: амортизация + электричество + поддержка."""
    amortization = hardware_price / amortization_months
    electricity = avg_power_watts / 1000 * 24 * 30 * kwh_price
    ops = ops_hours_per_month * engineer_hourly_rate
    return amortization + electricity + ops


def local_capacity_tokens_per_month(gen_tok_per_sec: float = 8.0,
                                    utilization: float = 0.5) -> float:
    """Сколько ВЫХОДНЫХ токенов в месяц способен выдать CPU-сервер.

    gen_tok_per_sec — скорость генерации (для 7-8B Q4 на CPU это 4-16 ток/с),
    utilization — доля времени под полезной нагрузкой (нагрузка неравномерна).
    """
    return gen_tok_per_sec * 3600 * 24 * 30 * utilization


print(f"Локальный CPU-сервер: ~${local_monthly_cost():.0f}/мес")
print(f"Его потолок: ~{local_capacity_tokens_per_month() / 1e6:.1f} млн выходных токенов/мес")

Теперь сравним три реальных профиля нагрузки при обоих ценовых уровнях.
Цены меняются каждый год — но **метод** расчёта останется тем же, просто подставьте
актуальные значения вашего провайдера.


In [ ]:
PRICES = {  # $ за 1 млн входных/выходных токенов — из таблицы урока 1.3
    "gpt-4o-mini":   (0.15, 0.60),   # бюджетный класс
    "claude-sonnet": (3.00, 15.00),  # топовый класс
}

scenarios = {
    "Чат-бот стартапа (200 диалогов/день)": Workload(200, 800, 300),
    "Классификация тикетов (5 000/день)": Workload(5_000, 1_500, 10),
    "Суммаризация документов (2 000/день)": Workload(2_000, 3_000, 400),
}

local_cost = local_monthly_cost()
capacity = local_capacity_tokens_per_month(gen_tok_per_sec=8.0)

print(f"{'Сценарий':<40} {'gpt-4o-mini':>12} {'claude-sonnet':>14} {'потолок CPU':>12}")
print("-" * 84)
for name, w in scenarios.items():
    cheap = api_monthly_cost(w, *PRICES["gpt-4o-mini"])
    top = api_monthly_cost(w, *PRICES["claude-sonnet"])
    fits = "влезает" if w.monthly_output_tokens <= capacity else "НЕ влезает!"
    print(f"{name:<40} {cheap:>10.0f}$/мес {top:>12.0f}$/мес {fits:>12}")

print(f"\nСвой CPU-сервер: ~${local_cost:.0f}/мес при потолке ~{capacity / 1e6:.1f} млн выходных токенов/мес")

Как читать результат (числа — из прогона выше):

- **При бюджетных ценах API выигрывает почти всегда.** Даже 5 000 классификаций
  в день — это ~$35/мес на `gpt-4o-mini` против ~$102/мес за свой сервер. Модели
  класса mini/flash подешевели настолько, что чистая экономика включается только
  на десятках тысяч запросов в день (точную границу посчитаем в упражнении 2).
- **При топовых ценах картина переворачивается**: та же классификация на
  `claude-sonnet` стоит ~$698/мес — локальный сервер в 7 раз дешевле. Вопрос лишь
  в том, хватает ли задаче локального качества, — это и будет измерять урок 5.2.
- **Суммаризация с длинными ответами упирается в физику**: нужно 24 млн выходных
  токенов в месяц, а потолок нашего CPU-сервера — ~10 млн. Тут не спасёт никакая
  цена: нужен GPU, API или несколько серверов.
- **Классификация — идеальный профиль для CPU**: выход крошечный (мало «дорогих»
  токенов генерации), а длинный вход обычно шаблонный — и его начало кешируется
  (подробности и живые замеры — в уроке 5.2). Чувствительные данные в тикетах — бонусом.

> Важно: расчёт игнорирует **качество**. Если локальная модель решает задачу на 70%,
> а облачная на 95% — сравнивать цены бессмысленно, пока не доказано, что 70% вам
> достаточно. Как это измерить — весь урок 5.2.


## Честные минусы локальных LLM

| Минус | Насколько это больно | Чем компенсируется |
|---|---|---|
| **Качество**: открытая 1–8B модель заметно слабее облачных гигантов в сложных рассуждениях, следовании инструкциям, редких языках | Больно на «умных» задачах: многошаговые рассуждения, сложный код, тонкая стилистика | Узкие задачи (классификация, извлечение, суммаризация) маленькие модели решают прилично; промпт-инжиниринг; дообучение (fine-tuning, модуль 6) |
| **Скорость на CPU**: 4–16 ток/с для 7–8B — это 30–90 секунд на абзац | Больно в интерактивных сценариях | Стриминг (пользователь видит текст сразу), маленькие модели, фоновая обработка очередями |
| **Операционная нагрузка**: обновления, мониторинг, OOM, версии моделей — теперь ваши проблемы | Постоянный налог времени | Уроки 5.3, 5.5 и 5.6 — ровно про это |
| **RAM**: 7–8B модель занимает 5–7 ГБ памяти в работе | На ноутбуке с 8 ГБ — впритык | Квантизация, модели 1–3B, выделенный сервер |

## Гибридные схемы — чаще всего побеждают они

Выбор «всё локально или всё в облаке» ложный. Рабочие паттерны:

- **Маршрутизация по чувствительности**: запросы с ПДн — в локальную модель, остальное — в API. Классический вариант для банков.
- **Маршрутизация по сложности**: простые запросы (80% трафика) — локально, сложные — в дорогую облачную модель.
- **Локально в dev/CI, облако в prod** — тесты бесплатные и детерминированные.
- **Черновик локально, проверка в облаке** (или наоборот — анонимизация локально, обработка в облаке).

## Матрица принятия решения

Чтобы решение не сводилось к «мне кажется», формализуем его в два шага:

1. **Жёсткие ограничения (hard constraints)** — фильтр «да/нет». Если комплаенс запрещает передачу данных — API исключён, счёт не важен. Если нужен GPT-класс качества и это доказано тестами — локальная 8B исключена.
2. **Взвешенная оценка (weighted scoring)** — по оставшимся критериям: каждому даём вес 1–5 (важность для вас) и оценку 1–5 каждому варианту.

Это тот же приём, что при выборе БД или брокера сообщений — только критерии другие.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Criterion:
    name: str
    weight: int       # 1..5 — важность критерия для ВАШЕЙ задачи
    local_score: int  # 1..5 — насколько хорош локальный вариант
    api_score: int    # 1..5 — насколько хорош облачный API


@dataclass
class Decision:
    verdict: str
    local_total: int = 0
    api_total: int = 0
    details: list[str] = field(default_factory=list)


def decide(criteria: list[Criterion],
           api_forbidden: bool = False,
           local_impossible: bool = False) -> Decision:
    """Двухшаговое решение: сначала жёсткие ограничения, потом взвешенный счёт."""
    if api_forbidden and local_impossible:
        return Decision("тупик: пересмотрите требования (например, гибрид с анонимизацией)")
    if api_forbidden:
        return Decision("ЛОКАЛЬНО (жёсткое ограничение: данные нельзя передавать вовне)")
    if local_impossible:
        return Decision("API (жёсткое ограничение: локальное качество/железо не тянет)")

    local_total = sum(c.weight * c.local_score for c in criteria)
    api_total = sum(c.weight * c.api_score for c in criteria)
    details = [
        f"  {c.name:<28} вес={c.weight}  локально={c.local_score}  api={c.api_score}"
        for c in criteria
    ]
    margin = abs(local_total - api_total) / max(local_total, api_total)
    if margin < 0.1:
        verdict = "ПАРИТЕТ (<10% разницы): выбирайте по команде/срокам или стройте гибрид"
    elif local_total > api_total:
        verdict = "склоняемся к ЛОКАЛЬНОЙ модели"
    else:
        verdict = "склоняемся к API"
    return Decision(verdict, local_total, api_total, details)


def print_decision(title: str, decision: Decision) -> None:
    print(f"=== {title} ===")
    for line in decision.details:
        print(line)
    if decision.local_total or decision.api_total:
        print(f"  ИТОГО: локально {decision.local_total} против api {decision.api_total}")
    print(f"  ВЕРДИКТ: {decision.verdict}\n")

Прогоним матрицу на трёх сценариях. Обратите внимание, как **веса** (а не оценки) меняются от задачи к задаче — оценки вариантов более-менее объективны, а вот важность критериев определяется вашим контекстом.

In [ ]:
# Сценарий 1: банковский чат-поддержки, в текстах ФИО и номера счетов.
# Комплаенс запрещает передачу наружу -> жёсткое ограничение, матрица не нужна.
print_decision(
    "Банк: классификация обращений с ПДн",
    decide(criteria=[], api_forbidden=True),
)

# Сценарий 2: MVP стартапа — ассистент по написанию писем, качество критично,
# трафик пока крошечный, команда — два человека без времени на ops.
startup = [
    Criterion("качество генерации",          weight=5, local_score=2, api_score=5),
    Criterion("скорость вывода",             weight=4, local_score=2, api_score=5),
    Criterion("стоимость при малом трафике", weight=3, local_score=3, api_score=5),
    Criterion("приватность данных",          weight=2, local_score=5, api_score=3),
    Criterion("нагрузка на команду (ops)",   weight=5, local_score=2, api_score=5),
]
print_decision("Стартап: ассистент писем (MVP)", decide(startup))

# Сценарий 3: приложение для геологов в полевых условиях — интернета нет вовсе.
print_decision(
    "Полевое приложение без интернета",
    decide(criteria=[], api_forbidden=True),  # offline = данные физически не могут уйти в API
)

# Сценарий 4: «Векторика» хочет суммаризовать логи инцидентов (3000 шт/день).
# Данные чувствительные (в постмортемах бывают ПДн — мы видели это в модуле 4),
# но не запрещённые; в команде есть DevOps.
internal = [
    Criterion("качество генерации",          weight=3, local_score=3, api_score=5),
    Criterion("скорость вывода",             weight=2, local_score=3, api_score=5),
    Criterion("стоимость при росте объёма",  weight=4, local_score=4, api_score=2),
    Criterion("приватность данных",          weight=5, local_score=5, api_score=2),
    Criterion("нагрузка на команду (ops)",   weight=3, local_score=3, api_score=5),
]
print_decision("Векторика: суммаризация логов инцидентов", decide(internal))

Матрица — не оракул, а **способ сделать спор предметным**: вместо «мне кажется, надо локально» вы обсуждаете конкретные веса и оценки. Если результат «паритет» — это тоже результат: значит, решают факторы за пределами таблицы (сроки, опыт команды) или стоит проектировать гибрид.

И финальный чек-лист урока — четыре вопроса, которые стоит задать до всякой матрицы:

1. Могут ли данные легально и по политике компании покидать контур? (нет → локально)
2. Должно ли это работать без интернета? (да → локально)
3. Доказано ли тестами, что открытая модель 1–8B решает задачу приемлемо? (нет → сначала урок 5.2)
4. Есть ли у команды ресурс на эксплуатацию ещё одного stateful-сервиса? (нет → API или managed-хостинг открытых моделей)

## ⚠️ Частые ошибки

1. **«Локально = бесплатно».** Бесплатны только веса. Платите вы железом, электричеством, RAM и — главное — временем инженеров на эксплуатацию. Считайте полную стоимость владения (TCO), как со своим Kubernetes против managed.
2. **Ожидать от 8B-модели уровня GPT/Claude.** Открытые маленькие модели отлично решают узкие задачи, но проигрывают в сложных рассуждениях. Сравнивайте на СВОИХ задачах (урок 5.2), а не на маркетинговых бенчмарках.
3. **«API-провайдеры обучаются на моих данных, поэтому только локально».** Для платных API у серьёзных провайдеров это по умолчанию не так (см. их DPA/политику). Реальные аргументы — комплаенс, ретеншн логов, юрисдикция и человеческий фактор. Стройте кейс на фактах, иначе его развалит первый же архитектурный комитет.
4. **Игнорировать гибрид.** «Всё локально» или «всё в API» — ложная дихотомия; маршрутизация по чувствительности/сложности часто даёт 80% пользы за 20% цены.
5. **Решать по цене за токен, забыв про потолок CPU.** Дешёвый локальный сервер, который физически не успевает генерировать нужный объём, — это не экономия, а инцидент.

## Упражнения

1. **Новый критерий.** Добавьте в матрицу критерий «предсказуемость задержки» (у локальной модели нет чужих rate limit, у API бывают всплески латентности) и пересчитайте сценарий «Векторика: суммаризация логов». Изменился ли вердикт?
2. **Точка безубыточности.** Напишите функцию `break_even_requests_per_day(workload_shape, price_in, price_out, local_cost)`, которая находит, при каком числе запросов в день API начинает стоить дороже локального сервера. Учтите потолок производительности: если нужный объём выходных токенов больше `local_capacity_tokens_per_month()`, верните `None` (одним CPU-сервером не обойтись).
3. **Классификация сценариев.** Для каждого из пяти сценариев ниже решите (сначала в голове, потом матрицей): локально, API или гибрид?
   - a) генерация описаний товаров для публичного маркетплейса, 50k шт разово;
   - b) медицинская выписка → структурированный JSON, клиника;
   - c) бот-помощник по документации open-source проекта;
   - d) анализ резюме кандидатов в HR-системе;
   - e) автодополнение кода во внутреннем IDE-плагине компании с NDA.

Решения — в конце ноутбука.

## Мини-квиз

<details><summary><b>1. Что из перечисленного — жёсткое ограничение, а не взвешиваемый критерий: а) стоимость токенов; б) запрет комплаенса на передачу данных вовне; в) скорость генерации?</b></summary>

**б.** Запрет комплаенса — бинарный фильтр: никакой вес других критериев его не
перевесит. Стоимость и скорость — взвешиваемые критерии: их можно «купить»
качеством по другим осям.
</details>

<details><summary><b>2. Почему задачи «длинный вход — короткий выход» особенно выгодны для локального CPU-инференса?</b></summary>

Три причины. Обработка входа (prefill) идёт батчем и на CPU в разы быстрее
генерации (а генерация упирается в пропускную способность памяти). Шаблонная часть
промпта (system + few-shot) кешируется и при повторных запросах почти бесплатна.
И короткий выход означает мало самых «дорогих» токенов — токенов генерации.
Живые замеры и разбор кеша префикса — в уроке 5.2.
</details>

<details><summary><b>3. Верно ли, что использование платного облачного API автоматически означает обучение провайдера на ваших данных?</b></summary>

**Нет.** У серьёзных провайдеров платный API по умолчанию не использует ваши данные
для обучения (это закреплено в договоре/DPA). Реальные риски — ретеншн логов,
юрисдикция, человеческий фактор — и именно их надо обсуждать с безопасниками.
Кейс «только локально», построенный на ложном аргументе, развалит первый же
архитектурный комитет.
</details>

<details><summary><b>4. Назовите две статьи расходов локального решения, кроме цены железа.</b></summary>

Электричество и время инженеров на эксплуатацию (мониторинг, обновления,
инциденты). Можно добавить третью: упущенная гибкость — железо купили, и оно
уже не масштабируется вниз в тихий месяц.
</details>

<details><summary><b>5. У вас 8 ГБ RAM на ноутбуке и задача классификации внутренних тикетов. С какой модели разумно начать: 8B в максимальном качестве или 1–3B?</b></summary>

**С 1–3B.** 8B даже в средней квантизации на 8 ГБ RAM будет жить впритык с ОС
и браузером (риск swap). Классификация — как раз задача, где 1–3B часто достаточно;
проверяется бенчмарком из урока 5.2.
</details>


## Итоги

- Локальная LLM — это веса на вашем диске и inference на вашем CPU: данные не покидают контур.
- Четыре драйвера локального запуска: приватность, комплаенс (152-ФЗ/GDPR/тайны), стоимость на объёме, offline/предсказуемость.
- Деньги считаются в обе стороны: у API — линейный рост с токенами, у локального сервера — фиксированная цена (~$102/мес в нашей модели), но и потолок производительности. С бюджетными облачными ценами экономика включается на десятках тысяч запросов в день, с топовыми — уже на сотнях.
- Честные минусы: качество ниже облачных гигантов, 4–16 ток/с на CPU для 7–8B, операционная нагрузка на вас.
- Решение принимается в два шага: жёсткие ограничения → взвешенная матрица. Гибридные схемы чаще всего оптимальны.

**Дальше**: урок 5.2 — как выбрать конкретную модель и квантизацию под вашу RAM и измерить качество на своих задачах, а не по лидербордам.

## Полезные ссылки

- Ollama: <https://ollama.com> и документация <https://github.com/ollama/ollama/tree/main/docs>
- llama.cpp (движок, на котором всё держится): <https://github.com/ggml-org/llama.cpp>
- Открытые модели: <https://huggingface.co/models> (фильтр по GGUF)
- Обзор трендов открытых моделей: <https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard> (помните про ограничения лидербордов!)

---

## Решения упражнений

### Упражнение 1: новый критерий

In [ ]:
internal_v2 = internal + [
    # Локальная модель: латентность стабильна (нет чужих лимитов), но сама по себе велика на CPU.
    # API: обычно быстрее, но бывают всплески и rate limit в час пик.
    Criterion("предсказуемость задержки", weight=3, local_score=5, api_score=3),
]
print_decision("Векторика: суммаризация логов + предсказуемость", decide(internal_v2))
# Вердикт сместился ещё сильнее в сторону локального варианта: критерий,
# по которому локальный вариант выигрывает, добавил ему очков.

### Упражнение 2: точка безубыточности

In [ ]:
def break_even_requests_per_day(input_tokens: int, output_tokens: int,
                                price_in: float, price_out: float,
                                local_cost: float | None = None,
                                gen_tok_per_sec: float = 8.0) -> int | None:
    """Минимальное число запросов/день, при котором API дороже локального сервера.

    None — если этот объём не влезает в потолок генерации одного CPU-сервера.
    """
    local_cost = local_cost if local_cost is not None else local_monthly_cost()
    cost_per_request = (input_tokens / 1e6 * price_in) + (output_tokens / 1e6 * price_out)
    if cost_per_request == 0:
        return None
    requests_per_month = local_cost / cost_per_request
    requests_per_day = int(requests_per_month / 30) + 1

    needed_output = requests_per_day * output_tokens * 30
    if needed_output > local_capacity_tokens_per_month(gen_tok_per_sec):
        return None  # одним CPU-сервером такой объём не отдать
    return requests_per_day


profiles = {
    "классификация (1500 -> 10)": (1500, 10),
    "чат (800 -> 300)": (800, 300),
    "суммаризация (3000 -> 400)": (3000, 400),
}

for tier, (price_in, price_out) in PRICES.items():
    print(f"--- цены {tier} ---")
    for name, (inp, out) in profiles.items():
        be = break_even_requests_per_day(inp, out, price_in, price_out)
        verdict = f"{be} запросов/день" if be else "недостижима на одном CPU-сервере"
        print(f"  {name:<28}: точка безубыточности = {verdict}")

# Мораль: с бюджетными ценами чат и суммаризация НЕ окупаются вовсе (нужный объём
# генерации выше потолка CPU), а классификация окупается с ~15 тыс. запросов в день.
# С топовыми ценами точка безубыточности падает до сотен запросов в день.

### Упражнение 3: классификация сценариев

- **a) Описания товаров для маркетплейса** — данные публичные, объём разовый, качество текста важно → **API** (или разовая аренда GPU, но это уже не про наш модуль). Локальный CPU будет генерировать 50k описаний неделями.
- **b) Медицинская выписка → JSON** — врачебная тайна, жёсткое ограничение → **локально**. Задача извлечения полей маленьким моделям по силам.
- **c) Бот по документации open-source** — данных чувствительных нет, трафик непредсказуем, качество ответов важно → **API**; локально — только из любви к искусству или для offline-версии.
- **d) Анализ резюме** — персональные данные кандидатов, объёмы средние → **локально или гибрид** (анонимизация локально → сложный анализ в API).
- **e) Автодополнение кода под NDA** — код — коммерческая тайна, задержка критична (нужен быстрый локальный инференс маленькой моделью 1–3B) → **локально**, причём скорее на машинах разработчиков.